# Lesson 06 Lab — Installing a Reproducible vLLM Environment

**Puzzle:** What evidence shows that Python, PyTorch, CUDA, the driver, and vLLM agree?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A successful package installation is not a successful GPU runtime. vLLM ships compiled components tied to platform and PyTorch choices, so the environment record must include imports, binary version, CUDA availability, GPU capability, and a minimal native operation.


## 0. Predict before running

1. Predict the compute capability and CUDA runtime reported remotely.
2. Check whether the vLLM CLI exposes serve and bench.
3. Name the minimal step beyond import needed for release confidence.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The compatibility probe imports vLLM and PyTorch, captures exact versions, locates the CLI, inspects selected engine arguments, and executes a CUDA tensor operation. It records missing features as data.

- Driver, wheel runtime, and compiler toolkit are different version fields.
- Import success is weaker than native engine execution.
- A pinned environment is part of every benchmark identity.


## 2. Derive the mechanism

The NVIDIA driver provides the kernel-facing CUDA capability; a PyTorch wheel carries its CUDA runtime; vLLM adds compiled extensions and generated kernels. These versions need not have identical labels, but the installed combination must support the GPU architecture and import without unresolved symbols. A clean environment prevents unrelated packages from silently replacing that combination.

### Mechanism at a glance

```mermaid
flowchart LR
  D["NVIDIA driver"] --> T["PyTorch CUDA runtime"]
  T --> V["vLLM compiled + Python package"]
  V --> M["model architecture + dtype"]
  M --> R["native generation"]
  R --> A["reproducible environment artifact"]
```

### Walk it step by step

1. **Pin the interpreter.** Create an isolated Python environment.
2. **Install one coherent stack.** Let the selected vLLM wheel resolve its compatible PyTorch build.
3. **Probe the executable path.** Verify imports, CLI, GPU identity, and a CUDA operation.
4. **Prove model execution.** Treat later native generation as the final compatibility link.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 6
LESSON_TITLE = 'Installing a Reproducible vLLM Environment'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260818
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | package metadata alone |
| Candidate | imports, CLI surface, compiled extension visibility, and CUDA execution |
| Held constant | isolated environment and one RTX 5090 |
| Measurements | versions, executable paths, CLI subcommands, tensor checksum, and feature flags |
| Evidence | `compatibility-probe` |

**Experiment:** Collect the complete stack identity and run a real CUDA sanity operation inside the isolated environment.


## 5. Inspect the experiment code

The probe avoids network downloads and shell-specific environment assumptions. Every field comes from the Python process that will execute the remaining labs.

Do not execute until the code matches the frozen table.


In [2]:
root_code,root_help=cli_help(); serve_code,serve_help=cli_help("serve"); bench_code,bench_help=cli_help("bench")
x=torch.arange(4096,device=DEVICE,dtype=torch.float32); checksum=float((x.sin()*x.cos()).sum().item())
metrics={"vllm_version":vllm.__version__,"python":sys.version.split()[0],"torch":torch.__version__,
         "cuda_runtime":str(torch.version.cuda),"cli_path":Path(vllm_cli()).name,"cli_found":Path(vllm_cli()).exists(),
         "serve_command":serve_code==0,"bench_command":bench_code==0,
         "cli_tokens":{"serve":"serve" in root_help.lower(),"bench":"bench" in root_help.lower()},
         "cuda_checksum":checksum}
analysis=(f"The isolated environment imported vLLM {vllm.__version__} with PyTorch {torch.__version__} "
          f"/ CUDA {torch.version.cuda}, found serve/bench={serve_code==0}/{bench_code==0}, and completed "
          f"a CUDA checksum of {checksum:.6f}. Native model generation is the stronger final link.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| vLLM version | 0.27.1 |
| Python | 3.12.3 |
| PyTorch | 2.13.0+cu130 |
| CUDA runtime | 13.0 |
| CLI found | yes |
| Serve command | yes |
| CUDA checksum | 0.352565 |


## 7. Explain the result

The isolated environment imported vLLM 0.27.1 with PyTorch 2.13.0+cu130 / CUDA 13.0, found serve/bench=True/True, and completed a CUDA checksum of 0.352565. Native model generation is the stronger final link.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 6, "title": 'Installing a Reproducible vLLM Environment', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Compatibility is a chain of executable checks; this probe establishes the local stack identity and CUDA path, not every model feature.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 6,
  "title": "Installing a Reproducible vLLM Environment",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260818
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "vllm_version": "0.27.1",
    "python": "3.12.3",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "cli_path": "vllm",
    "cli_found": true,
    "serve_command": true,
    "bench_command": true,
    "cli_tokens": {
      "serve": true,
      "bench": true
    },
    "cuda_checksum": 0.3525652289390564
  },
  "analysis": "The isolated environment imported vLLM 0.27.1 with PyTorch 2.13.0+cu130 / CUDA 13.0, found serve/bench=True/True, and completed a CUDA checksum of 0.352565. Native model generation is the stronger final link.",
  "conclusion": "Compatibility is a chain of executabl

## 9. Make the bounded decision

> Compatibility is a chain of executable checks; this probe establishes the local stack identity and CUDA path, not every model feature.

**Acceptance/rollback:** Proceed to model experiments only when the pinned interpreter imports the stack, sees the GPU, and completes a CUDA operation.

**Failure analysis:** A small tensor operation exercises PyTorch rather than every vLLM kernel. Later model loads can still fail because of architecture, dtype, memory, or compilation issues.


## 10. Extend the evidence

Archive `pip freeze`, vLLM collect-env output, driver information, model hash, and one completed model-generation artifact with the release.

The full boundary and references are in [`README.md`](README.md).
